In [8]:
import tkinter as tk
from tkinter import ttk, messagebox
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# =========================
# Load CSV
# =========================
df = pd.read_csv("Expanded_Destinations.csv")
df.fillna("", inplace=True)

# Standardize column names (strip spaces and capitalize)
df.columns = [c.strip().title() for c in df.columns]
print("Columns after standardizing:", df.columns)


Columns after standardizing: Index(['Destinationid', 'Name', 'State', 'Type', 'Popularity',
       'Besttimetovisit', 'Reviewtext', 'Visitdate', 'Experiencerating',
       'Preferences'],
      dtype='object')


In [9]:
# =========================
# Combine text for embeddings
# =========================
def combine_text_features(df):
    df["combined_text"] = (
        df["Name"].astype(str) + " " +
        df["Reviewtext"].astype(str) + " " +
        df["Preferences"].astype(str)
    )
    return df

df = combine_text_features(df)

# =========================
# Normalize popularity score
# =========================
def normalize_scores(df):
    scaler = MinMaxScaler()
    df["popularity_norm"] = scaler.fit_transform(df[["Popularity"]])
    return df

df = normalize_scores(df)


In [10]:
# =========================
# Load SentenceTransformer model
# =========================
model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings for combined text
embeddings = model.encode(df["combined_text"].tolist(), show_progress_bar=True)


Batches: 100%|██████████| 47/47 [00:04<00:00, 10.56it/s]


In [11]:
# =========================
# Recommendation function
# =========================
def recommend_destinations(query):
    if not query.strip():
        messagebox.showwarning("Input Error", "Please enter a travel preference.")
        return

    sim_weight = similarity_slider.get() / 100
    pop_weight = popularity_slider.get() / 100

    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, embeddings)[0]

    df["similarity"] = similarities
    df["final_score"] = sim_weight * df["similarity"] + pop_weight * df["popularity_norm"]

    # Remove duplicate destinations (by Name)
    results = df.sort_values("final_score", ascending=False)\
                .drop_duplicates(subset="Name", keep="first")\
                .head(5)

    display_results(results)

# =========================
# Display results
# =========================
def display_results(results):
    for widget in result_frame.winfo_children():
        widget.destroy()

    for _, row in results.iterrows():
        card = tk.Frame(result_frame, bg="white", bd=1, relief="solid")
        card.pack(fill="x", pady=8, padx=10)

        tk.Label(card, text=row["Name"], font=("Segoe UI", 13, "bold"), bg="white", fg="#0b57c1").pack(anchor="w", padx=10, pady=4)
        tk.Label(card, text=row["Reviewtext"][:180]+"...", wraplength=600, justify="left", bg="white").pack(anchor="w", padx=10)
        tk.Label(card, text=f"Match Score: {row['final_score']:.2f}", bg="white", fg="gray").pack(anchor="e", padx=10, pady=4)

# =========================
# GUI Setup
# =========================
root = tk.Tk()
root.title("AI Travel Recommendation System")
root.geometry("800x600")
root.configure(bg="#f4f6f9")

# Title
tk.Label(root, text="🌍 Smart Travel Recommendation", font=("Segoe UI", 18, "bold"), bg="#f4f6f9").pack(pady=10)

# Input Box
input_frame = tk.Frame(root, bg="#f4f6f9")
input_frame.pack(pady=5)

query_entry = ttk.Entry(input_frame, width=50, font=("Segoe UI", 11))
query_entry.pack(side="left", padx=10)
query_entry.insert(0, "Beach, adventure, peaceful...")

ttk.Button(input_frame, text="Recommend", command=lambda: recommend_destinations(query_entry.get())).pack(side="left")

# Sliders
slider_frame = tk.Frame(root, bg="#f4f6f9")
slider_frame.pack(pady=10)

tk.Label(slider_frame, text="Similarity", bg="#f4f6f9").grid(row=0, column=0)
similarity_slider = ttk.Scale(slider_frame, from_=0, to=100, value=70)
similarity_slider.grid(row=0, column=1, padx=10)

tk.Label(slider_frame, text="Popularity", bg="#f4f6f9").grid(row=1, column=0)
popularity_slider = ttk.Scale(slider_frame, from_=0, to=100, value=30)
popularity_slider.grid(row=1, column=1, padx=10)

# Scrollable Results
canvas = tk.Canvas(root, bg="#f4f6f9")
scrollbar = ttk.Scrollbar(root, orient="vertical", command=canvas.yview)
scroll_frame = tk.Frame(canvas, bg="#f4f6f9")

scroll_frame.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))

canvas.create_window((0,0), window=scroll_frame, anchor="nw")
canvas.configure(yscrollcommand=scrollbar.set)

canvas.pack(side="left", fill="both", expand=True)
scrollbar.pack(side="right", fill="y")

result_frame = scroll_frame




In [12]:
root.mainloop()
